In [1]:
transactions = [
    {'milk', 'bread', 'butter'},
    {'bread', 'butter'},
    {'milk', 'bread'},
    {'milk', 'bread', 'butter', 'eggs'},
    {'bread', 'butter', 'eggs'}
]

total_transactions = len(transactions)

def support(itemset):
    count = 0
    for t in transactions:
        if itemset.issubset(t):
            count += 1
    return count / total_transactions

def confidence(antecedent, consequent):
    count_antecedent = 0
    count_both = 0
    for t in transactions:
        if antecedent.issubset(t):
            count_antecedent += 1
            if consequent.issubset(t):
                count_both += 1
    if count_antecedent == 0:
        return 0
    return count_both / count_antecedent

def lift(antecedent, consequent):
    conf = confidence(antecedent, consequent)
    supp_consequent = support(consequent)
    if supp_consequent == 0:
        return 0
    return conf / supp_consequent

# Milk -> Butter
antecedent = {'milk'}
consequent = {'butter'}
s = support(antecedent.union(consequent))
c = confidence(antecedent, consequent)
l = lift(antecedent, consequent)
print(f"Milk -> Butter: Support = {s:.4f}, Confidence = {c:.4f}, Lift = {l:.4f}")

# Milk -> Bread
antecedent = {'milk'}
consequent = {'bread'}
s = support(antecedent.union(consequent))
c = confidence(antecedent, consequent)
l = lift(antecedent, consequent)
print(f"Milk -> Bread: Support = {s:.4f}, Confidence = {c:.4f}, Lift = {l:.4f}")

# Milk, Bread -> Butter
antecedent = {'milk', 'bread'}
consequent = {'butter'}
s = support(antecedent.union(consequent))
c = confidence(antecedent, consequent)
l = lift(antecedent, consequent)
print(f"Milk, Bread -> Butter: Support = {s:.4f}, Confidence = {c:.4f}, Lift = {l:.4f}")

Milk -> Butter: Support = 0.4000, Confidence = 0.6667, Lift = 0.8333
Milk -> Bread: Support = 0.6000, Confidence = 1.0000, Lift = 1.0000
Milk, Bread -> Butter: Support = 0.4000, Confidence = 0.6667, Lift = 0.8333


In [2]:
# apriori
import itertools

transactions = [
    {'milk', 'bread', 'butter'},
    {'bread', 'butter'},
    {'milk', 'bread'},
    {'milk', 'bread', 'butter', 'eggs'},
    {'bread', 'butter', 'eggs'}
]

min_support = 0.6
total_transactions = len(transactions)

def get_candidates(freq_itemsets, k):
    candidates = set()
    for i in range(len(freq_itemsets)):
        for j in range(i+1, len(freq_itemsets)):
            union = freq_itemsets[i] | freq_itemsets[j]
            if len(union) == k:
                candidates.add(frozenset(union))
    return list(candidates)

def apriori(transactions, min_support):
    items = set()
    for t in transactions:
        items |= t
    freq_itemsets = [frozenset([item]) for item in items]
    result = {}
    k = 1
    while freq_itemsets:
        freq_itemsets_counts = {}
        for candidate in freq_itemsets:
            count = sum(1 for t in transactions if candidate.issubset(t))
            support = count / total_transactions
            if support >= min_support:
                freq_itemsets_counts[candidate] = support
        if not freq_itemsets_counts:
            break
        result.update(freq_itemsets_counts)
        freq_itemsets = get_candidates(list(freq_itemsets_counts.keys()), k+1)
        k += 1
    return result

freq_itemsets = apriori(transactions, min_support)
print("Frequent Itemsets from Apriori:")
for itemset, support in freq_itemsets.items():
    print(f"{set(itemset)}: {support:.2f}")


Frequent Itemsets from Apriori:
{'bread'}: 1.00
{'milk'}: 0.60
{'butter'}: 0.80
{'bread', 'milk'}: 0.60
{'bread', 'butter'}: 0.80


In [3]:
# FP-Growth
class FPNode:
    def __init__(self, item, count=1):
        self.item = item
        self.count = count
        self.parent = None
        self.children = {}
        self.next = None

def build_fp_tree(transactions, min_support):
    from collections import defaultdict, Counter
    
    total_transactions = len(transactions)
    item_counts = Counter()
    for t in transactions:
        item_counts.update(t)
    item_counts = {k: v for k, v in item_counts.items() if v / total_transactions >= min_support}
    if not item_counts:
        return None, None

    header_table = {k: None for k in item_counts}
    
    def insert_tree(items, node):
        first = items[0]
        if first in node.children:
            node.children[first].count += 1
        else:
            new_node = FPNode(first)
            new_node.parent = node
            node.children[first] = new_node
            
            if header_table[first] is None:
                header_table[first] = new_node
            else:
                current = header_table[first]
                while current.next:
                    current = current.next
                current.next = new_node
        if len(items) > 1:
            insert_tree(items[1:], node.children[first])
            
    root = FPNode(None)
    for t in transactions:
        sorted_items = [item for item in sorted(t, key=lambda x: item_counts.get(x,0), reverse=True) if item in item_counts]
        if sorted_items:
            insert_tree(sorted_items, root)
    return root, header_table

fp_root, header_table = build_fp_tree(transactions, min_support)
print("\nFP-Tree built. Header table keys (frequent items):", list(header_table.keys()))



FP-Tree built. Header table keys (frequent items): ['bread', 'milk', 'butter']


In [6]:
# distance based
from scipy.spatial import distance

items = sorted(set(it for t in transactions for it in t))

def transaction_to_vector(transaction, items):
    return [1 if item in transaction else 0 for item in items]

vectors = [transaction_to_vector(t, items) for t in transactions]

print("\nItems:", items)
print("Transaction vectors:")
for i, vec in enumerate(vectors, 1):
    print(f"T{i}:", vec)

# Example: Distance between T1 and T2
v1 = vectors[0]
v2 = vectors[1]

euclidean_dist = distance.euclidean(v1, v2)
manhattan_dist = distance.cityblock(v1, v2)
chebyshev_dist = distance.chebyshev(v1, v2)

print(f"\nDistance between T1 and T2:")
print(f"Euclidean: {euclidean_dist}")
print(f"Manhattan: {manhattan_dist}")
print(f"Chebyshev: {chebyshev_dist}")


Items: ['bread', 'butter', 'eggs', 'milk']
Transaction vectors:
T1: [1, 1, 0, 1]
T2: [1, 1, 0, 0]
T3: [1, 0, 0, 1]
T4: [1, 1, 1, 1]
T5: [1, 1, 1, 0]

Distance between T1 and T2:
Euclidean: 1.0
Manhattan: 1
Chebyshev: 1


In [20]:
import pandas as pd

transactions = {
    "T0": {'milk', 'bread', 'butter'},
    "T1": {'bread', 'butter'},
    "T2": {'milk', 'bread'},
    "T3": {'milk', 'bread', 'butter', 'eggs'},
    "T4": {'bread', 'butter', 'eggs'}
}

# Convert to a list of dicts with TransactionID and comma-separated items
data = []
for tid, items in transactions.items():
    data.append({"TransactionID": tid, "Items": ",".join(items)})

df = pd.DataFrame(data)

# Save to CSV
df.to_csv("transactions.csv", index=False)

print(df)


  TransactionID                   Items
0            T0       bread,milk,butter
1            T1            bread,butter
2            T2              bread,milk
3            T3  eggs,bread,milk,butter
4            T4       eggs,bread,butter


In [23]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Sample transaction data (similar to your example)
transactions = [
    ['milk', 'bread', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['bread', 'butter', 'eggs']
]

# Get unique items
all_items = sorted(set(item for transaction in transactions for item in transaction))

# One-hot encode transactions into a DataFrame with only 0/1
def encode_transaction(transaction, items):
    return [1 if item in transaction else 0 for item in items]

onehot_encoded = [encode_transaction(t, all_items) for t in transactions]
onehot_df = pd.DataFrame(onehot_encoded, columns=all_items)

fp = fpgrowth(onehot_df, min_support=0.2, use_colnames=True)

# Generate association rules with confidence >= 0.7
rules = association_rules(fp, metric="confidence", min_threshold=0.7)

print("Frequent Itemsets:")
print(fp)

print("\nAssociation Rules:")
print(rules)


Frequent Itemsets:
    support                     itemsets
0       1.0                      (bread)
1       0.8                     (butter)
2       0.6                       (milk)
3       0.4                       (eggs)
4       0.8              (bread, butter)
5       0.6                (bread, milk)
6       0.4               (milk, butter)
7       0.4        (bread, milk, butter)
8       0.4               (eggs, butter)
9       0.4                (eggs, bread)
10      0.2                 (eggs, milk)
11      0.4        (eggs, bread, butter)
12      0.2         (eggs, milk, butter)
13      0.2          (eggs, bread, milk)
14      0.2  (eggs, bread, milk, butter)

Association Rules:
             antecedents      consequents  antecedent support  \
0                (bread)         (butter)                 1.0   
1               (butter)          (bread)                 0.8   
2                 (milk)          (bread)                 0.6   
3         (milk, butter)          (bread)    

/home/cse-lab/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/home/cse-lab/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)
